# Lab 10 — Interactive Cluster Visualizer

Labs 04 and 06 used raw `docker exec` commands to simulate drive failures.
This lab wraps those mechanics in an **interactive ipywidgets dashboard** —
click buttons to fail/restore drives and observe EC reconstruction live.

### What you will see

```
┌─────────────────────────────────────────────────────────┐
│  RustFS Cluster — RS(4,2)                                │
│                                                          │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐   │
│  │ 🟢 Drive 0│ │ 🟢 Drive 1│ │ 🔴 Drive 2│ │ 🟢 Drive 3│   │
│  │  ONLINE  │ │  ONLINE  │ │  FAILED  │ │  ONLINE  │   │
│  └──────────┘ └──────────┘ └──────────┘ └──────────┘   │
│                                                          │
│  [Fail Drive 0] [Fail Drive 1] [Fail Drive 2] ...        │
│  [Restore All]  [Read Objects]                           │
│                                                          │
│  Status: RS(4,2) — 5/6 shards online — DEGRADED ✓       │
│                                                          │
│  ── Activity Log ─────────────────────────────────────── │
│  [12:34:01] Drive 2 failed                               │
│  [12:34:03] GET obj_001 → RECONSTRUCTED (5 shards)       │
│  [12:34:03] GET obj_002 → RECONSTRUCTED (5 shards)       │
└─────────────────────────────────────────────────────────┘
```


---
## Setup — Load Data

In [ ]:
import sys
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, '../scripts')
from lab_utils import (
    cleanup_bucket,
    ensure_bucket,
    fail_drive,
    get_s3_client,
    restore_all_drives,
)

s3 = get_s3_client()
BUCKET = 'lab10-viz'
CONTAINER = 'rustfs-server'
NUM_DRIVES = 4
NUM_OBJECTS = 8

ensure_bucket(s3, BUCKET)

# Upload test objects
OBJECTS = {f'obj_{i:03d}': f'Content of object {i} — verify this after drive failures!'.encode()
           for i in range(NUM_OBJECTS)}

for key, val in OBJECTS.items():
    s3.put_object(Bucket=BUCKET, Key=key, Body=val)

print(f'✅ Loaded {NUM_OBJECTS} objects into s3://{BUCKET}/')
print('Scroll down to launch the dashboard.')

---
## Interactive Dashboard

In [ ]:
# ── State ────────────────────────────────────────────────────────────────────
failed_drives: set[int] = set()


def ts() -> str:
    return datetime.now().strftime('%H:%M:%S')


# ── Widgets ──────────────────────────────────────────────────────────────────

# Drive status boxes (HTML labels)
drive_labels = [widgets.HTML() for _ in range(NUM_DRIVES)]

# Fail buttons
fail_buttons = [
    widgets.Button(
        description=f'Fail Drive {i}',
        button_style='danger',
        layout=widgets.Layout(width='120px'),
    )
    for i in range(NUM_DRIVES)
]

restore_btn = widgets.Button(
    description='Restore All',
    button_style='success',
    icon='refresh',
    layout=widgets.Layout(width='130px'),
)

read_btn = widgets.Button(
    description='Read All Objects',
    button_style='primary',
    icon='download',
    layout=widgets.Layout(width='150px'),
)

status_label = widgets.HTML()
log_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ccc',
        height='200px',
        overflow_y='auto',
        padding='8px',
        font_family='monospace',
    )
)


# ── Render helpers ───────────────────────────────────────────────────────────

def render_drives():
    online = NUM_DRIVES - len(failed_drives)
    for i, label in enumerate(drive_labels):
        if i in failed_drives:
            label.value = (
                f'<div style="border:2px solid #e74c3c; border-radius:6px; '
                f'padding:8px; background:#fdf2f2; text-align:center; width:110px">'
                f'<b>🔴 Drive {i}</b><br><span style="color:#e74c3c; font-size:11px">FAILED</span></div>'
            )
            fail_buttons[i].disabled = True
        else:
            label.value = (
                f'<div style="border:2px solid #2ecc71; border-radius:6px; '
                f'padding:8px; background:#f2fdf6; text-align:center; width:110px">'
                f'<b>🟢 Drive {i}</b><br><span style="color:#27ae60; font-size:11px">ONLINE</span></div>'
            )
            fail_buttons[i].disabled = False

    # EC status
    k = 4  # data shards required
    if len(failed_drives) == 0:
        state, color = 'HEALTHY ✅', '#27ae60'
    elif len(failed_drives) <= (NUM_DRIVES - k):
        state, color = f'DEGRADED ⚠️  ({len(failed_drives)} drive(s) offline — EC reconstructing)', '#e67e22'
    else:
        state, color = f'CRITICAL ❌  ({len(failed_drives)} drives offline — DATA UNRECOVERABLE)', '#e74c3c'

    status_label.value = (
        f'<div style="padding:8px; background:#f8f9fa; border-radius:4px; margin:4px 0">'
        f'<b>RS(4,2) Cluster Status:</b> '
        f'<span style="color:{color}; font-weight:bold">{state}</span>'
        f'&nbsp;&nbsp;|&nbsp;&nbsp;'
        f'<b>{online}/{NUM_DRIVES}</b> drives online'
        f'</div>'
    )


def log(msg: str):
    with log_output:
        print(f'[{ts()}] {msg}')


# ── Event handlers ───────────────────────────────────────────────────────────

def on_fail_drive(drive_idx):
    def handler(btn):
        if drive_idx in failed_drives:
            return
        fail_drive(CONTAINER, BUCKET, drive=drive_idx)
        failed_drives.add(drive_idx)
        log(f'Drive {drive_idx} FAILED')
        render_drives()
    return handler


def on_restore(btn):
    restore_all_drives(CONTAINER, BUCKET, NUM_DRIVES)
    failed_drives.clear()
    log('All drives restored')
    render_drives()


def on_read(btn):
    ok = 0
    fail = 0
    for key, expected in OBJECTS.items():
        try:
            body = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
            status = 'RECONSTRUCTED' if failed_drives else 'OK'
            assert body == expected
            ok += 1
            log(f'GET {key} → {status} ({NUM_DRIVES - len(failed_drives)}/4 data shards available)')
        except Exception as e:
            fail += 1
            log(f'GET {key} → ERROR: {e}')
    log(f'Read complete: {ok} OK, {fail} FAILED')


# Wire up handlers
for i, btn in enumerate(fail_buttons):
    btn.on_click(on_fail_drive(i))
restore_btn.on_click(on_restore)
read_btn.on_click(on_read)

# ── Layout ───────────────────────────────────────────────────────────────────

title = widgets.HTML('<h3 style="margin:0">🗄️ RustFS Cluster Dashboard — RS(4,2)</h3>')
drive_row = widgets.HBox(drive_labels, layout=widgets.Layout(gap='10px', margin='8px 0'))
fail_row = widgets.HBox(fail_buttons, layout=widgets.Layout(gap='8px'))
action_row = widgets.HBox([restore_btn, read_btn], layout=widgets.Layout(gap='8px', margin='4px 0'))
log_title = widgets.HTML('<b>── Activity Log ────────────────────────────────</b>')

dashboard = widgets.VBox([
    title,
    drive_row,
    status_label,
    fail_row,
    action_row,
    log_title,
    log_output,
], layout=widgets.Layout(padding='16px', border='1px solid #dee2e6', border_radius='8px'))

render_drives()
log(f'Cluster ready — {NUM_OBJECTS} objects loaded into s3://{BUCKET}/')
display(dashboard)

---
## How to Use This Dashboard

| Action | Expected result |
|--------|-----------------|
| Click **Read All Objects** | All 8 objects return OK (full cluster healthy) |
| Click **Fail Drive 0**, then **Read All Objects** | All objects return RECONSTRUCTED — EC repairs from 3 remaining drives |
| Click **Fail Drive 1** (now 2 failed), then **Read All** | Still RECONSTRUCTED — RS(4,2) survives 2 concurrent failures |
| Click **Fail Drive 2** (now 3 failed), then **Read All** | ❌ ERRORS — below k=4 threshold, data is unrecoverable |
| Click **Restore All**, then **Read All Objects** | All objects return OK again |


---
## Cleanup

In [ ]:
# Make sure all drives are restored before deleting
restore_all_drives(CONTAINER, BUCKET, NUM_DRIVES)
cleanup_bucket(s3, BUCKET)
print('✅ Cluster restored and bucket cleaned up')